# Scaling benchmarks on the Colab T4

Runs the three headline scaling sweeps and syncs each result to Drive the
moment it finishes (`MyDrive/jax-marl-bc-runs/benchmarks/<name>/`):

| # | sweep | what it measures | figures |
|---|-------|------------------|---------|
| 1 | `scaling_paper_ks` | KS, **1 env**, n = 10..500 at the paper's budget (1e5 per-agent transitions) vs the original MARL-BC single-CPU times (Gabriele et al. 2026, Fig. 8, digitized in `configs/reference/`) | walltime, throughput, **speedup** |
| 2 | `scaling_phase` | 8x8 mesh of n_agents x num_envs at fixed sequential budget; every n*E = 200 pair lies on the grid | **phase diagram**, tradeoff cut |
| 3 | `scaling_agents` | 5 envs, n_agents 1 -> 20000 on a 1-2-5 progression | walltime, throughput |

Each sweep writes `benchmarks/<name>/{results.csv,sweep.yaml,*.png}`; the raw
timing table is the record, figures can be regenerated ex post. Rough T4
budget: ~15-30 min each — run cells top to bottom, or any sweep on its own
(setup + Drive cells first).

In [ ]:
# Setup: clone or update the repo, install (idempotent — safe to re-run).
%cd /content
![ -d jax-marl-bc ] || git clone https://github.com/danmonuni/jax-marl-bc.git
%cd jax-marl-bc
!git pull
!pip install -q -r requirements.txt && pip install -q -e . --no-deps

In [ ]:
# Sanity: a GPU runtime is attached (Runtime > Change runtime type > T4 GPU).
!nvidia-smi -L

In [ ]:
# Mount Drive BEFORE the sweeps so each benchmark is saved as soon as it
# finishes (a Colab disconnect then loses at most the sweep in progress,
# never a finished one).
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

def save_benchmark(name):
    """Sync benchmarks/<name> -> Drive (exact path, idempotent re-sync)."""
    src = f'benchmarks/{name}'
    dst = f'/content/drive/MyDrive/jax-marl-bc-runs/benchmarks/{name}'
    assert os.path.exists(src), f"{src} missing - did the sweep finish?"
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"saved {src} -> {dst}")

## Sweep 1/3 — head-to-head vs the original paper

KS economy, single parallel env, n = 10, 20, 50, 100, 200, 500 at the matched
budget (`total_timesteps=1e5` sequential steps = 1e5 transitions per agent =
the paper's "1e5 per-agent updates"). One timed run per cell with the compile
phase split out, so `time_s` is steady-state only. The speedup figure divides
each digitized CPU series (PPO is like-for-like) by our time at the same n.

In [ ]:
!python -m jmbc.sweep sweep=scaling_paper_ks

In [ ]:
save_benchmark('scaling-paper-ks')

## Sweep 2/3 — agents x envs phase diagram

Full 8x8 mesh over n_agents, num_envs in {1, 2, 5, 10, 20, 40, 100, 200} at a
fixed sequential budget (100 updates/cell; one timed run each — the AOT
phase timer splits compile from run). Produces phase diagrams of absolute run time and of total
transitions/s throughput plus the
constant-product tradeoff cut along n_agents * num_envs = 200.

In [ ]:
!python -m jmbc.sweep sweep=scaling_phase

In [ ]:
save_benchmark('scaling-phase')

## Sweep 3/3 — population scaling at 5 envs

n_agents = 1 -> 20000 on the 1-2-5 progression, num_envs = 5, fixed sequential
budget. Shows how far the T4 absorbs batch width (flat wall time) before it
saturates. Largest cell forecasts ~4.5 GB device memory.

In [ ]:
!python -m jmbc.sweep sweep=scaling_agents

In [ ]:
save_benchmark('scaling-agents-5env')

## Headline figures

In [ ]:
import os
from IPython.display import Image, display
for p in [
    'benchmarks/scaling-paper-ks/walltime_vs_n_agents.png',
    'benchmarks/scaling-paper-ks/speedup_vs_n_agents.png',
    'benchmarks/scaling-phase/phase_time.png',
    'benchmarks/scaling-phase/phase_throughput.png',
    'benchmarks/scaling-phase/tradeoff_product200.png',
    'benchmarks/scaling-agents-5env/walltime_vs_n_agents.png',
    'benchmarks/scaling-agents-5env/throughput_vs_n_agents.png',
]:
    if os.path.exists(p):
        print(p)
        display(Image(p))
    else:
        print(f"(missing: {p})")